# From texts to variables, with and without an LLM

Session 4. The professions de foi of the first round of the **1978 legislative election**: Archelec (Sciences Po, CEVIPOF), through the Internet Archive API. One question runs through the notebook:

> **Do candidates who already hold a local office talk more about local issues?**

| Part | What you do | What you get |
| --- | --- | --- |
| 1 | Collect 300 documents and their catalogue records | the texts, and the candidates' current offices (written by Sciences Po) |
| 2 | Code 6 documents by hand | the class's gold standard |
| 3 | A dictionary, then embeddings and a classifier | two measures without an LLM |
| 4 | An LLM fills in a form for each document, Claude and GPT | a table, checked field by field against the catalogue |
| 5 | From labels to an estimate | the coefficient, naive and corrected |
| 6 | Keep what a referee would ask for | a bundle to download |
| 7 | Your own 100 texts (optional) | the same measures on your project |

**Before you start.** Colab Secrets (key icon, left sidebar): `ANTHROPIC_API_KEY` and `OPENAI_API_KEY`, "Notebook access" on. Never paste a key into a cell. A CPU runtime is enough; a T4 GPU (Runtime, Change runtime type, *before* running anything) makes Part 3 faster.

**On the ScienceCluster** (ScienceApps, Jupyter, kernel `s4`): keys in `~/.env`; skip the install cell; set `CLUSTER_EMBEDDINGS` to the folder written by `embed.sbatch`.

The texts are under a restrictive licence (CC BY-NC-ND): keep them out of any public repository.

## 0. Setup (run once; nothing to explain here)

In [ ]:
%pip install -q anthropic openai pydantic sentence-transformers scikit-learn statsmodels

In [ ]:
import datetime, hashlib, json, os, pathlib, random, re, shutil, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from pydantic import BaseModel, ConfigDict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_predict

# ---- Parameters: everything a referee would ask about is here ----
DATE = "1978-03-12"            # first round of the 1978 legislative election (4,010 professions de foi in Archelec)
N = 300                        # documents drawn at random from that election. Keep N and SEED as they are:
SEED = 2026                    # the class's hand codes refer to this draw (and to the order of hand coding)
MY_NUMBER = 0                  # YOUR number, from Hugo's email (1, 2, 3...). 0 = not set yet
GOLD_SHEET_CSV = ""            # the class's hand codes: the CSV link of the shared sheet (in the shared document)
CLAUDE_MODEL = "claude-haiku-5-5"
GPT_MODEL = "gpt-6-luna"
EMBED_MODEL = "intfloat/multilingual-e5-small"   # MIT licence, 384 numbers per passage, French included
CLUSTER_EMBEDDINGS = ""        # ScienceApps only: e.g. "/scratch/<shortname>/archelec/1978-03-12/embeddings"
MAX_CHARS = 15000              # longer texts are cut before they go to a model (the count is printed)
PRICE = {"in": 0.10, "out": 0.50, "cache_read": 0.01, "cache_write": 0.125}  # $ per million tokens, both models, Nov. 2026

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M")
ROOT = pathlib.Path("s4_work")
RAW = ROOT / "data" / "raw" / "archelec" / DATE     # raw answers of the API, never edited
LLM = ROOT / "outputs" / "llm"                       # every model answer, appended as it arrives
RUN = ROOT / "outputs" / RUN_ID
for p in (RAW / "texts", LLM, RUN):
    p.mkdir(parents=True, exist_ok=True)
print("Run:", RUN)

In [ ]:
# Plumbing: keys, requests with retries, text normalisation, scores. Read it once if you like.

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    for f in (pathlib.Path(".env"), pathlib.Path.home() / ".env"):
        if f.exists():
            for line in f.read_text().splitlines():
                if line.strip().startswith(name + "="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None


HEADERS = {"User-Agent": "uzh-llm-socsci-2026 (course notebook)"}


def get(url, params=None, tries=5):
    """GET with retries. Raises after the last try: a silent failure is the worst kind."""
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(2 ** i)
    raise RuntimeError(f"failed after {tries} tries: {url}")


def fetch(identifier):
    """The catalogue record and the OCR text of one document. Rerun-safe: texts on disk are not downloaded again."""
    txt = RAW / "texts" / f"{identifier}.txt"
    m = get(f"https://archive.org/metadata/{identifier}")
    meta = (m.json().get("metadata") or None) if m is not None else None
    if not txt.exists():
        r = get(f"https://archive.org/download/{identifier}/{identifier}_djvu.txt")
        if r is not None:
            txt.write_text(r.text, encoding="utf-8")
    return identifier, meta


def norm(s):
    """Lower case, no accents, no punctuation: 'Député-Maire' -> 'depute maire'."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[’`]", "'", s)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]", " ", s)).strip()


EMPTY = {"", "nr", "non precise", "none", "nan", "?"}   # "NR" = not recorded in the catalogue


def empty(v):
    return norm(v) in EMPTY


def flat(v):
    return " / ".join(map(str, v)) if isinstance(v, list) else ("" if v is None else str(v))


def score(pred, truth):
    """A measure against a reference, on the documents both have."""
    d = pd.concat([pd.Series(pred).rename("p"), pd.Series(truth).rename("t")], axis=1).dropna().astype(int)
    if d.empty:
        return {"n": 0}
    return {"n": len(d), "yes (measure)": int(d.p.sum()), "yes (reference)": int(d.t.sum()),
            "accuracy": round(accuracy_score(d.t, d.p), 2),
            "precision": round(precision_score(d.t, d.p, zero_division=0), 2),
            "recall": round(recall_score(d.t, d.p, zero_division=0), 2),
            "kappa": round(cohen_kappa_score(d.t, d.p), 2) if d.t.nunique() > 1 or d.p.nunique() > 1 else None}


print("ready")

## 1. Collect: a data API, again

Internet Archive's search API. The same four questions as in Session 3: what did I ask for, what came back, what is missing, and against which number do I check?

In [ ]:
QUERY = f'collection:archiveselectoralesducevipof AND date:{DATE} AND type:"profession de foi"'
r = get("https://archive.org/advancedsearch.php",
        {"q": QUERY, "fl[]": "identifier", "sort[]": "identifier asc", "rows": 10000, "output": "json"})
FOUND = r.json()["response"]
FOUND["numFound"], len(FOUND["docs"])

Check 1: the search returned every document it found. If not, it is paginating, and we read page one only.

In [ ]:
ALL_IDS = [d["identifier"] for d in FOUND["docs"]]
assert len(ALL_IDS) == FOUND["numFound"], "the search is paginating: ask for more rows"
(RAW / "ids.json").write_text(json.dumps({"query": QUERY, "numFound": FOUND["numFound"], "ids": ALL_IDS}))
ALL_IDS[:3]

An identifier reads like a code: election, year, month, *département*, constituency, round, document. A random draw of `N`, fixed by `SEED`.

In [ ]:
SAMPLE = sorted(random.Random(SEED).sample(ALL_IDS, N))
len(SAMPLE), SAMPLE[0]

In [ ]:
with ThreadPoolExecutor(8) as pool:
    FETCHED = dict(pool.map(fetch, SAMPLE))
TEXT = {i: (RAW / "texts" / f"{i}.txt").read_text(encoding="utf-8")
        for i in SAMPLE if (RAW / "texts" / f"{i}.txt").exists()}
missing = [i for i in SAMPLE if FETCHED.get(i) is None or i not in TEXT]
print(f"requested {len(SAMPLE)} = returned {len(SAMPLE) - len(missing)} + missing {len(missing)}", missing[:5])

Check 2 and 3: no empty text, no duplicate. An OCR that failed gives a text of a few characters.

In [ ]:
lengths = pd.Series({i: len(t) for i, t in TEXT.items()})
short = lengths[lengths < 300].index.tolist()
dupes = pd.Series({i: hashlib.md5(t.encode()).hexdigest() for i, t in TEXT.items()}).duplicated().sum()
print("characters per text:", lengths.describe()[["min", "50%", "max"]].round().to_dict(),
      "| shorter than 300:", len(short), "| duplicates:", dupes, "| longer than MAX_CHARS:", int((lengths > MAX_CHARS).sum()))

One document. Open the PDF next to it: what does the OCR get wrong?

In [ ]:
print(f"https://archive.org/details/{SAMPLE[0]}\n")
print(TEXT[SAMPLE[0]][:1500])

### The catalogue: someone else's coding

Every document has a record written by Sciences Po: the candidate, the substitute (*suppléant*), the profession, the current offices, the party. "NR" means *not recorded*. This is the number we did not produce: Part 4 checks the LLM against it.

In [ ]:
FIELDS = {"nom": "ba-nom-titulaire", "prenom": "bb-prenom-titulaire", "sexe": "bc-sexe-titulaire",
          "profession": "be-profession-titulaire", "mandat": "bf-mandat-en-cours-titulaire",
          "soutien": "bj-soutien-titulaire", "liste": "bk-liste-titulaire",
          "supp_nom": "ca-nom-suppleant", "supp_profession": "ce-profession-suppleant",
          "departement": "departement", "circonscription": "df-circonscription"}
CAT = pd.DataFrame({i: {k: flat(m.get(v)) for k, v in FIELDS.items()} for i, m in FETCHED.items() if m}).T
CAT.head()

**X, the explanatory variable**: does the candidate hold a local office now (mayor, deputy mayor, municipal, general or regional councillor)? From the catalogue. Sitting deputies are flagged separately.

In [ ]:
LOCAL = re.compile(r"\bmaire\b|\badjoint|conseil\w* (municipal|general|regional|departemental|d'arrondissement|de paris)"
                   r"|president du conseil (general|regional|departemental)")


def is_local(m):
    return int(bool(LOCAL.search(norm(m))))


def is_deputy(m):
    return int(bool(re.search(r"\bdepute", norm(m))))


CAT["local_mandate"] = CAT.mandat.map(is_local)
CAT["deputy"] = CAT.mandat.map(is_deputy)
KEEP = [i for i in SAMPLE if i in TEXT and i in CAT.index and i not in short]
CAT = CAT.loc[KEEP]
print(len(KEEP), "documents kept |", CAT.local_mandate.mean().round(2), "hold a local office |",
      CAT.deputy.mean().round(2), "are sitting deputies")
CAT.mandat[CAT.local_mandate == 1].value_counts().head(8)

## 2. Code by hand, before any model

**Y, the outcome**: does the document make a *local offer*? The definition is the one of the Session 3 codebook, word for word:

> **local_offer**: a geographically specific local issue, institution, project, service, firm, infrastructure, place, or population. A candidate name, constituency heading, or place name alone is insufficient.

Everyone codes the same first document, then five of their own, drawn at random. Set `MY_NUMBER` in Part 0 first, and rerun that cell.

In [ ]:
ORDER = random.Random(SEED + 1).sample(SAMPLE, len(SAMPLE))   # the order in which documents are handed out
COMMON = ORDER[0]


def my_docs(k):
    return ORDER[1 + 5 * (k - 1): 1 + 5 * k]


MINE = [COMMON] + (my_docs(MY_NUMBER) if MY_NUMBER else [])
for i in MINE:
    print(i, f"https://archive.org/details/{i}")

Read each document (the PDF is easier than the OCR). Then write 1 (a local offer) or 0 (none) next to each identifier.

In [ ]:
# Your codes, in the order printed above: the common document first, then your five. Replace each None by 1 or 0.
MY_CODES = dict(zip(MINE, [None, None, None, None, None, None]))
MY_CODES

Paste these lines into the shared sheet (one row per document). The sheet is the class's gold standard.

In [ ]:
print("\n".join(f"{MY_NUMBER},{i},{v}" for i, v in MY_CODES.items() if v is not None))

### The class's codes

When the sheet is full, Hugo gives its CSV link. Put it in `GOLD_SHEET_CSV` (Part 0) and run the next cell. Without it, the notebook uses your own codes only (or `gold_local_offer.csv`, if Hugo published one).

In [ ]:
def read_codes():
    g = None
    if GOLD_SHEET_CSV:
        try:
            g = pd.read_csv(GOLD_SHEET_CSV)
        except Exception as e:
            print("Could not read the sheet (is it shared 'anyone with the link'?):", repr(e)[:120], "| falling back to gold_local_offer.csv, or your own codes")
    if g is not None:
        pass
    elif pathlib.Path("gold_local_offer.csv").exists():
        g = pd.read_csv("gold_local_offer.csv")
    else:
        g = pd.DataFrame([{"participant": MY_NUMBER, "doc_id": i, "local_offer": v}
                          for i, v in MY_CODES.items() if v is not None])
    g["doc_id"] = g.doc_id.astype(str).str.strip()
    g["local_offer"] = pd.to_numeric(g.local_offer, errors="coerce")
    g["participant"] = pd.to_numeric(g.participant, errors="coerce").fillna(0).astype(int)
    g = g[g.local_offer.isin([0, 1]) & g.doc_id.isin(KEEP)].copy()
    # Only documents that were handed out: the gold sample must stay random.
    handed = set(ORDER[:1 + 5 * max(1, int(g.participant.max()) if len(g) else 1)])
    dropped = (~g.doc_id.isin(handed)).sum()
    if dropped:
        print(f"{dropped} code(s) on documents that were not handed out: dropped")
    g = g[g.doc_id.isin(handed)]
    g["local_offer"] = g.local_offer.astype(int)
    return g


CODES = read_codes()
GOLD = CODES.groupby("doc_id").local_offer.mean().ge(0.5).astype(int)   # several coders: the majority
R = pd.Series(KEEP, index=KEEP).isin(GOLD.index).astype(int)             # 1 = hand-coded
PI = R.mean()                                                             # the probability of being hand-coded
print(len(CODES), "codes from", CODES.participant.nunique(), "coders on", len(GOLD), "documents |",
      "share with a local offer:", round(GOLD.mean(), 2) if len(GOLD) else None)

**Do we agree?** The common document, coded by everyone.

In [ ]:
CODES[CODES.doc_id == COMMON].local_offer.value_counts()

The gold documents are a **random sample** of the 300 (the handing-out order was drawn at random), with a known probability of being coded: their number over the total. Part 5 needs exactly that.

In [ ]:
print(f"{R.sum()} of {len(R)} documents hand-coded: probability {PI:.3f}")

## 3. Measuring without an LLM

### 3a. A dictionary

A list of words, and a rule. Everything is visible: you can print it in an appendix, and it gives the same answer every time.

In [ ]:
LOCAL_WORDS = ["notre ville", "notre commune", "nos communes", "nos villages", "notre canton", "notre region",
               "notre departement", "la circonscription", "route", "rocade", "deviation", "autoroute", "pont",
               "gare", "voie ferree", "hopital", "maternite", "college", "lycee", "piscine", "usine",
               "zone industrielle", "port", "canal", "amenagement", "desenclavement", "assainissement"]
THRESHOLD = 2   # a document is coded 1 when it contains at least THRESHOLD different words of the list


def hits(text, words=None):
    words = LOCAL_WORDS if words is None else words
    t = " " + norm(text) + " "
    return [w for w in words if f" {w} " in t]

In [ ]:
DOCS = pd.DataFrame(index=KEEP)
DOCS["dict_hits"] = [len(hits(TEXT[i])) for i in KEEP]
DOCS["dict"] = (DOCS.dict_hits >= THRESHOLD).astype(int)
DOCS.dict.mean().round(2)

In [ ]:
score(DOCS.dict, GOLD)

Where it is wrong. A false positive: which words fired, and in what context?

In [ ]:
def kwic(text, words=None, width=60):
    words, t = (LOCAL_WORDS if words is None else words), norm(text)
    return [t[max(0, m.start() - width): m.end() + width] for w in words for m in re.finditer(rf"\b{w}\b", t)][:6]


false_pos = [i for i in GOLD.index if DOCS.dict[i] == 1 and GOLD[i] == 0]
false_neg = [i for i in GOLD.index if DOCS.dict[i] == 0 and GOLD[i] == 1]
print(len(false_pos), "false positives,", len(false_neg), "false negatives")
kwic(TEXT[false_pos[0]]) if false_pos else "no false positive"

**Exercise.** Remove a word that fires on headings (`"la circonscription"`?), add two words you saw in a false negative, rerun the three cells above. Precision and recall move in opposite directions: that is the dictionary's trade-off, in plain sight.

### 3b. Embeddings: meaning as coordinates

Each passage of 80 words becomes a point in a space of 384 dimensions; passages that say similar things are close, even with other words, or with OCR errors.

In [ ]:
from sentence_transformers import SentenceTransformer

EMB = SentenceTransformer(EMBED_MODEL)   # downloads once (about 470 MB); uses the GPU if there is one
print(EMB.device)

In [ ]:
def passages(text, words=80):
    w = re.sub(r"\s+", " ", text).split(" ")
    return [" ".join(w[i:i + words]) for i in range(0, len(w), words) if len(w[i:i + words]) >= 10]


if CLUSTER_EMBEDDINGS:                   # computed by embed.sbatch on the cluster: read them, do not recompute
    P_all = pd.read_csv(pathlib.Path(CLUSTER_EMBEDDINGS) / "passages.csv")
    V_all = np.load(pathlib.Path(CLUSTER_EMBEDDINGS) / "passages.npy").astype(np.float32)
    keep = P_all.doc_id.isin(KEEP).values
    P, PV = P_all[keep].reset_index(drop=True), V_all[keep]
else:
    P = pd.DataFrame([(i, j, s) for i in KEEP for j, s in enumerate(passages(TEXT[i]))],
                     columns=["doc_id", "passage", "text"])
    t0 = time.time()
    PV = EMB.encode(["query: " + t for t in P.text], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    print(f"{len(P)} passages in {time.time() - t0:.0f} seconds")
PV.shape

**Search by meaning.** The five passages closest to a sentence you write.

In [ ]:
def search(sentence, k=5):
    v = EMB.encode(["query: " + sentence], normalize_embeddings=True)[0]
    s = PV @ v
    top = np.argsort(-s)[:k]
    return P.iloc[top].assign(similarity=s[top].round(3))[["similarity", "doc_id", "text"]]


pd.set_option("display.max_colwidth", 200)
search("Il faut sauver la maternité de l'hôpital de notre ville.")

A score per document: its passage closest to a few examples of local offers. No threshold yet, so we read it as a ranking: the AUC is the probability that a document with a local offer ranks above one without.

In [ ]:
PROTOS = ["Je me battrai pour la déviation de la route nationale qui traverse notre ville.",
          "Il faut maintenir la maternité et les urgences de l'hôpital de notre commune.",
          "J'obtiendrai l'implantation d'une usine dans la zone industrielle du canton.",
          "La ligne de chemin de fer de la vallée ne doit pas fermer."]
proto = EMB.encode(["query: " + s for s in PROTOS], normalize_embeddings=True).mean(0)
P["proto"] = PV @ (proto / np.linalg.norm(proto))
DOCS["proto"] = P.groupby("doc_id").proto.max().reindex(KEEP)
g = GOLD.index.intersection(DOCS.index)
round(roc_auc_score(GOLD[g], DOCS.proto[g]), 2) if GOLD[g].nunique() == 2 else "need both 0 and 1 in the gold codes"

### 3c. A classifier trained on your codes

The document's vector (the mean of its passages) as input, the class's codes as labels, a logistic regression. Scored by **cross-validation**: each gold document is predicted by a model that never saw it.

In [ ]:
DV = (pd.DataFrame(PV, index=P.doc_id).groupby(level=0).mean()).reindex(KEEP)
DV = DV.div(np.linalg.norm(DV.values, axis=1), axis=0)
yg = GOLD[GOLD.index.intersection(DV.index)]
folds = int(min(5, yg.value_counts().min())) if yg.nunique() == 2 else 0
print(len(yg), "gold documents,", folds, "folds")

In [ ]:
CLF = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000)
if folds >= 2:
    oof = pd.Series(cross_val_predict(CLF, DV.loc[yg.index], yg, cv=folds), index=yg.index)
    CLF.fit(DV.loc[yg.index], yg)
    DOCS["clf"] = CLF.predict(DV.loc[KEEP])
    DOCS.loc[yg.index, "clf"] = oof          # gold documents keep the prediction of a model that did not see them
    print(score(oof, yg))
else:
    print("Not enough gold codes yet (at least two of each). Come back after the sheet is filled.")

**When the classical tools win.** On 4,010 documents, the dictionary costs nothing and the classifier costs a minute of GPU (see `embed.sbatch`); both give the same answer tomorrow; neither sends a text anywhere. Their weakness: the dictionary reads words, not meaning, and the classifier needs a few hundred codes to be good.

## 4. Extracting with an LLM: from a text to a table

Classification gives one label per document. **Extraction** fills in a form: who is the candidate, what is their profession, which offices do they hold, who supports them, who is the substitute. Plus one box that is a judgement, not a copy: `local_offer`, with the same definition as in Part 2.

The form is a schema: the model cannot answer outside it. An empty field is an answer; an invented one is an error.

In [ ]:
class Person(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nom: str
    prenom: str
    sexe: Literal["H", "F", "?"]
    profession: str
    mandats_en_cours: list[str]


class LocalOffer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    quote: str          # the evidence first, then the decision
    present: bool


class Form(BaseModel):
    model_config = ConfigDict(extra="forbid")
    titulaire: Person
    soutien: str
    suppleant: Person
    local_offer: LocalOffer

In [ ]:
INSTRUCTIONS = """You read the campaign statement (profession de foi) of one candidate in a French legislative election, from an OCR scan. Fill in the form. The French text is authoritative. It may contain OCR errors: read through them, but never guess what cannot be read. Leave a field empty rather than guess: an empty field is an answer, an invented value is an error.

titulaire: the candidate the document is for.
suppleant: the candidate's substitute (suppléant, remplaçant). If the document names none, leave all its fields empty ("" and an empty list) and set sexe to "?".

For each person:
- nom, prenom: the surname and first name as printed, without titles (M., Mme, Docteur, Maître).
- sexe: "H" or "F" when the text shows it (Madame, Monsieur, a feminine or masculine form, the first name); "?" if you cannot tell.
- profession: the occupation as printed, for example "agriculteur", "ouvrier", "professeur", "médecin". Not a party position, not an elected office. "" if none is given.
- mandats_en_cours: every elected office the person holds at the time of the election, as printed, for example "maire de Carvin", "conseiller général du canton de Lens", "député sortant". Not past offices ("ancien maire"), not party, union or association positions. An empty list if none.

soutien: the party, movement or coalition that presents or supports the candidate, as printed, for example "Parti communiste français", "Union pour la démocratie française", "RPR". "" if none is named.

local_offer: a geographically specific local issue, institution, project, service, firm, infrastructure, place, or population. A candidate name, constituency heading, or place name alone is insufficient.
- quote: one verbatim French quote of at most 15 words that shows it; "" if absent. Write the quote before you decide.
- present: true or false."""

SCHEMA = Form.model_json_schema()
CONFIG = hashlib.sha256((INSTRUCTIONS + json.dumps(SCHEMA, sort_keys=True)).encode()).hexdigest()[:8]
print(len(INSTRUCTIONS.split()), "words of instructions | config", CONFIG)

### Claude

In [ ]:
import anthropic

assert get_secret("ANTHROPIC_API_KEY"), "No key: add ANTHROPIC_API_KEY in Colab Secrets (key icon), Notebook access on"
claude = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"), max_retries=5)


def extract_claude(text):
    r = claude.messages.parse(
        model=CLAUDE_MODEL, max_tokens=1500,
        system=[{"type": "text", "text": INSTRUCTIONS, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": text[:MAX_CHARS]}],
        output_format=Form,
        thinking={"type": "disabled"},          # a well-defined task: no thinking, low cost
    )
    u = r.usage
    return {"form": r.parsed_output.model_dump() if r.parsed_output else None, "stop": r.stop_reason,
            "model_returned": r.model, "in": u.input_tokens, "out": u.output_tokens,
            "cache_read": u.cache_read_input_tokens or 0, "cache_write": u.cache_creation_input_tokens or 0}

One document. Read everything that comes back before you run 300 of them, and compare with the catalogue record.

In [ ]:
first = extract_claude(TEXT[KEEP[0]])
first

In [ ]:
CAT.loc[KEEP[0], ["nom", "prenom", "sexe", "profession", "mandat", "soutien", "liste", "supp_nom"]].to_dict()

In [ ]:
def run(fn, name, workers=4):
    """Every document once. Answers are appended to a JSONL file as they arrive: a crash or a rerun never pays twice.
    The file name carries the model and a hash of the instructions and schema: change them, and you get fresh answers."""
    path = LLM / f"{name}_{CONFIG}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text().splitlines():
            rec = json.loads(line)
            if rec.get("form"):
                done[rec["doc_id"]] = rec
    todo = [i for i in KEEP if i not in done]

    def one(i):
        t0 = time.time()
        try:
            rec = fn(TEXT[i])
        except Exception as e:
            rec = {"form": None, "error": repr(e)[:300]}
        rec.update(doc_id=i, seconds=round(time.time() - t0, 2),
                   utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
        return rec

    with ThreadPoolExecutor(workers) as pool, path.open("a") as f:
        for k, rec in enumerate(pool.map(one, todo), 1):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            done[rec["doc_id"]] = rec
            if k % 50 == 0:
                print(f"  {name}: {k}/{len(todo)}")
    out = pd.DataFrame([done.get(i, {"doc_id": i, "form": None}) for i in KEEP]).set_index("doc_id")
    n_err = out.form.isna().sum()
    print(f"{name}: {len(out) - n_err}/{len(out)} forms" + (f", {n_err} errors: rerun the cell" if n_err else ""))
    return out


def cost(res):
    tok = res[["in", "out", "cache_read", "cache_write"]].fillna(0).sum()
    return round(sum(tok[k] * PRICE[k] for k in PRICE) / 1e6, 3)

In [ ]:
RESULTS = {"claude": run(extract_claude, "claude_" + CLAUDE_MODEL)}
print("cost: $", cost(RESULTS["claude"]))

### GPT: the same form, the same instructions

In [ ]:
from openai import OpenAI

gpt = OpenAI(api_key=get_secret("OPENAI_API_KEY") or "missing", max_retries=5)


def extract_gpt(text):
    r = gpt.responses.parse(model=GPT_MODEL, max_output_tokens=1500, instructions=INSTRUCTIONS,
                            input=text[:MAX_CHARS], text_format=Form, reasoning={"effort": "none"})
    u = r.usage
    cached = getattr(getattr(u, "input_tokens_details", None), "cached_tokens", 0) or 0
    return {"form": r.output_parsed.model_dump() if r.output_parsed else None, "stop": r.status,
            "model_returned": r.model, "in": u.input_tokens - cached, "out": u.output_tokens,
            "cache_read": cached, "cache_write": 0}


if get_secret("OPENAI_API_KEY"):
    RESULTS["gpt"] = run(extract_gpt, "gpt_" + GPT_MODEL)
    print("cost: $", cost(RESULTS["gpt"]))
else:
    print("No OPENAI_API_KEY: GPT skipped. Everything below works with Claude alone.")

### The table

In [ ]:
def tidy(res):
    rows = {}
    for i, f in res.form.items():
        if not f:
            continue
        t, s = f["titulaire"], f["suppleant"]
        rows[i] = {"nom": t["nom"], "prenom": t["prenom"], "sexe": t["sexe"], "profession": t["profession"],
                   "mandat": " / ".join(t["mandats_en_cours"]), "soutien": f["soutien"],
                   "supp_nom": s["nom"], "supp_profession": s["profession"],
                   "local_offer": int(f["local_offer"]["present"]), "quote": f["local_offer"]["quote"]}
    return pd.DataFrame.from_dict(rows, orient="index")


EXT = {m: tidy(r) for m, r in RESULTS.items()}
EXT["claude"].head()

### Field by field, against the catalogue

Five outcomes per field: **match**, **mismatch**, **model only** (the catalogue says NR, the model gives a value: an invention, or a catalogue gap?), **catalogue only** (the model left it empty), **both empty**. Names must be equal; professions and parties may differ in wording ("ouvrier surface" and "Ouvrier de la surface"; "PCF" and "Parti communiste français").

In [ ]:
STOP = {"de", "la", "le", "les", "des", "du", "d", "l", "et", "en", "au", "aux", "a"}


def tokens(s):
    return {w for w in norm(s).replace("'", " ").split() if w not in STOP}


def acronym(s):
    return "".join(w[0] for w in norm(s).replace("'", " ").split() if w not in STOP)


def same(a, b, field):
    if field in ("nom", "prenom", "sexe", "supp_nom"):
        a, b = norm(a), norm(b)
        return a == b or (min(len(a), len(b)) > 3 and (a in b or b in a))
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return False
    return ta <= tb or tb <= ta or len(ta & tb) / len(ta | tb) >= 0.5 or acronym(a) in tb or acronym(b) in ta


def outcome(a, refs, field):
    refs = [b for b in refs if not empty(b)]
    if empty(a):
        return "catalogue only" if refs else "both empty"
    if not refs:
        return "model only"
    return "match" if any(same(a, b, field) for b in refs) else "mismatch"


def compare(ext, field):
    refs = (lambda i: [CAT.at[i, "soutien"], CAT.at[i, "liste"]]) if field == "soutien" else (lambda i: [CAT.at[i, field]])
    return pd.Series({i: outcome(ext.at[i, field], refs(i), field) for i in ext.index})


CHECKED = ["nom", "prenom", "sexe", "profession", "soutien", "supp_nom", "supp_profession"]
OUT = {m: pd.DataFrame({f: compare(e, f) for f in CHECKED}) for m, e in EXT.items()}
TABLE = pd.concat({m: o.apply(lambda c: c.value_counts()).fillna(0).astype(int) for m, o in OUT.items()})
TABLE

In [ ]:
# Accuracy where both have a value
pd.DataFrame({m: (o == "match").sum() / o.isin(["match", "mismatch"]).sum() for m, o in OUT.items()}).round(2)

**Read five disagreements.** Who is right: the model, or the catalogue? Open the PDF.

In [ ]:
def disagreements(model, k=5):
    o = OUT[model].stack()
    o = o[o.isin(["mismatch", "model only"])]
    cat = lambda i, f: f"{CAT.at[i, 'soutien']} | {CAT.at[i, 'liste']}" if f == "soutien" else CAT.at[i, f]
    rows = [{"doc_id": i, "field": f, "outcome": v, "model": EXT[model].at[i, f],
             "catalogue": cat(i, f), "pdf": f"https://archive.org/details/{i}"}
            for (i, f), v in o.sample(min(k, len(o)), random_state=SEED).items()]
    return pd.DataFrame(rows)


disagreements("claude")

The offices: the model's list turned into the same 0/1 as the catalogue's, and crossed with it.

In [ ]:
for m, e in EXT.items():
    x = e.mandat.map(is_local).rename(m)
    print(pd.crosstab(CAT.local_mandate.reindex(x.index).rename("catalogue"), x), "\n")

**Putting it together.** One row per candidate: the catalogue's variables, the extracted ones, and four measures of the local offer. This is a dataset you can merge with election results by *département* and constituency.

In [ ]:
DATA = CAT[["departement", "circonscription", "local_mandate", "deputy"]].copy()
DATA["dict"], DATA["proto"] = DOCS.dict, DOCS.proto.round(3)
if "clf" in DOCS:
    DATA["clf"] = DOCS.clf
for m, e in EXT.items():
    DATA[m] = e.local_offer.reindex(DATA.index)
    DATA[m + "_mandate"] = e.mandat.map(is_local).reindex(DATA.index)
DATA["gold"] = GOLD.reindex(DATA.index)
DATA.head()

Every measure against the class's codes. The question we left open in Session 3 starts here.

In [ ]:
MEASURES = [m for m in ["dict", "clf", "claude", "gpt"] if m in DATA]   # proto is a score, not a 0/1: not here
pd.DataFrame({m: score(DATA[m], DATA.gold) for m in MEASURES}).T

## 5. From labels to an estimate

> The model agrees with the human coders on 75% of the documents. Your regression uses its labels. What would you have to show for a referee to accept the coefficient?

Errors in a 0/1 label do not wash out. Unrelated to X, they shrink the gap; related to X, they can push it either way: if the model over-codes local offers among mayors (who name their town more often), the gap is inflated. A random gold sample, coded by hand with a known probability, corrects it (Egami et al. 2023, *design-based supervised learning*):

$$\tilde Y_i = \hat Y_i + \frac{R_i}{\pi}\,\big(Y_i - \hat Y_i\big)$$

$\hat Y_i$: the model's label; $Y_i$: the hand code, observed when $R_i=1$; $\pi$: the probability of being hand-coded. Then regress $\tilde Y$ on X. Three lines.

In [ ]:
def ols(y, d):
    return smf.ols(f"{y} ~ local_mandate", data=d).fit(cov_type="HC1")


def corrected(measure):
    d = DATA.dropna(subset=[measure]).copy()
    r_ = d.gold.notna().astype(int)
    d["pseudo"] = d[measure] + r_ / PI * (d.gold.fillna(0) - d[measure])
    return ols("pseudo", d)

In [ ]:
rows = {"hand codes only": ols("gold", DATA.dropna(subset=["gold"]))} if DATA.gold.notna().sum() >= 10 else {}
for m in MEASURES:
    rows[f"{m}, naive"] = ols(m, DATA.dropna(subset=[m]))
    if DATA.gold.notna().sum() >= 10:
        rows[f"{m}, corrected"] = corrected(m)
EST = pd.DataFrame({k: {"coef": f.params["local_mandate"], "low": f.conf_int().loc["local_mandate", 0],
                        "high": f.conf_int().loc["local_mandate", 1], "n": int(f.nobs)} for k, f in rows.items()}).T
EST.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.45 * len(EST) + 1))
y = np.arange(len(EST))[::-1]
ax.errorbar(EST.coef, y, xerr=[EST.coef - EST.low, EST.high - EST.coef], fmt="o", color="#3333B2", capsize=3)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(y, EST.index)
ax.set_xlabel("Local office holders minus others: share making a local offer (95% CI)")
plt.tight_layout()
plt.savefig(RUN / "estimates.png", dpi=150)

**How to read it.** Naive and corrected agree: you can say so, and the referee has a reason to believe the labels. They disagree: the corrected one is the estimate, and the gap is the bias of the labels. The corrected interval is wider: the price of not trusting the model.

**What goes in the paper.** The codebook; the model ID and the date; agreement with the hand codes on a random sample; the sampling probability; the corrected estimate and its interval; the raw answers in the replication files.

## 6. Keep what a referee would ask for

In [ ]:
(RUN / "instructions.txt").write_text(INSTRUCTIONS)
(RUN / "schema.json").write_text(json.dumps(SCHEMA, indent=2))
for f in LLM.glob(f"*_{CONFIG}.jsonl"):
    shutil.copy(f, RUN / f.name)
DATA.to_csv(RUN / "dataset.csv")                  # variables only, no text
CODES.to_csv(RUN / "hand_codes.csv", index=False)
EST.to_csv(RUN / "estimates.csv")
TABLE.to_csv(RUN / "extraction_vs_catalogue.csv")
params = {
    "run_id": RUN_ID, "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "data": {"source": "Archelec (Sciences Po, CEVIPOF) via Internet Archive", "query": QUERY,
             "numFound": FOUND["numFound"], "sample": N, "kept": len(KEEP), "seed": SEED,
             "ids_sha256": hashlib.sha256(json.dumps(KEEP).encode()).hexdigest()},
    "hand_codes": {"coders": int(CODES.participant.nunique()), "documents": int(len(GOLD)), "probability": round(PI, 4),
                   "sampling": "random order drawn with SEED + 1; first documents of that order"},
    "dictionary": {"words": LOCAL_WORDS, "threshold": THRESHOLD},
    "embeddings": {"model": EMBED_MODEL, "words_per_passage": 80, "from_cluster": bool(CLUSTER_EMBEDDINGS),
                   "prototypes": PROTOS},
    "llm": {"claude": CLAUDE_MODEL, "gpt": GPT_MODEL, "thinking": "disabled / reasoning effort none",
            "max_chars": MAX_CHARS, "instructions_sha256": hashlib.sha256(INSTRUCTIONS.encode()).hexdigest(),
            "config": CONFIG, "cost_usd": {m: cost(r) for m, r in RESULTS.items()}},
}
(RUN / "params.json").write_text(json.dumps(params, indent=2, ensure_ascii=False))
print(sorted(p.name for p in RUN.iterdir()))

In [ ]:
shutil.make_archive(str(RUN), "zip", RUN)
try:
    from google.colab import files
    files.download(f"{RUN}.zip")
except ImportError:
    print("Bundle:", f"{RUN}.zip")

## 7. Your own texts (optional)

A CSV with a column `text` and a column `label` (0 or 1): the 100 texts you coded yourself. Upload it (Colab: Files panel), write your definition and a few words, run the cells. Open-ended survey answers work the same way.

In [ ]:
MY_FILE = ""                 # e.g. "my_texts.csv"
MY_DEFINITION = """label: ..."""   # your codebook entry, as precise as the one for local_offer
MY_WORDS = []                # a few words, normalised (lower case, no accents)

In [ ]:
class Label(BaseModel):
    model_config = ConfigDict(extra="forbid")
    quote: str
    present: bool


def claude_label(text):
    r = claude.messages.parse(model=CLAUDE_MODEL, max_tokens=300, thinking={"type": "disabled"},
                              system=MY_DEFINITION + "\nquote: a verbatim quote of at most 15 words, \"\" if absent. present: true or false.",
                              messages=[{"role": "user", "content": text[:MAX_CHARS]}], output_format=Label)
    return int(r.parsed_output.present) if r.parsed_output else None


if MY_FILE:
    T = pd.read_csv(MY_FILE).dropna(subset=["text", "label"])
    T["label"] = T.label.astype(int)
    T["dict"] = [int(len(hits(t, MY_WORDS)) >= 1) for t in T.text]
    V = EMB.encode(["query: " + t[:2000] for t in T.text], normalize_embeddings=True)
    T["clf"] = cross_val_predict(LogisticRegression(class_weight="balanced", max_iter=1000), V, T.label,
                                 cv=int(min(5, T.label.value_counts().min())))
    with ThreadPoolExecutor(8) as pool:
        T["claude"] = list(pool.map(claude_label, T.text))
    print(pd.DataFrame({m: score(T[m], T.label) for m in ["dict", "clf", "claude"]}).T)

**What you just did.** A data API with its checks; a gold standard coded by hand, at random; three ways to measure the same concept (a word list, a geometry, a model) and one to extract a table; and a coefficient a referee can accept. The prompts 12 to 14 in `template/prompts/` do the same on your project, with your agent.